# Chapter 4: Building an Autograd Engine from First Principles

Welcome to **Chapter 4**!

In Chapters 1, 2, and 3, whenever we added a new layer (like Dense layers or Conv2D), we had to sit down with a piece of paper, work out the calculus chain rule by hand, and write custom `backward()` methods.

### But what happens in modern AI?
Today's models (Transformers, GPT-4, Diffusion Models) contain hundreds of complex operations — attention mechanisms, layer norms, residual additions, and rotary embeddings. Deriving and maintaining manual derivatives for every new architecture is **impossible to scale and fraught with subtle math bugs**.

### The Solution: Automatic Differentiation (Autograd)
Libraries like **PyTorch** and **JAX** don't write manual backward passes for each model. Instead, they provide an **Autograd Engine**:
1. **Forward Pass**: As you perform math operations, the engine quietly constructs a **Directed Acyclic Graph (DAG)** of all tensors and operations.
2. **Backward Pass**: When you call `.backward()`, the engine automatically performs a **topological sort** of the graph and traverses it in reverse order, applying the chain rule to compute exact gradients for every single variable!

In this notebook, we build our own PyTorch-like Autograd engine in pure NumPy from scratch.

---

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure repository root is on sys.path
repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.autograd import Tensor, Parameter, Module, Linear, MLP, cross_entropy_loss

print("Chapter 4 Autograd Engine loaded successfully from:", repo_root)

## 1. A Simple Computational Graph: Forward & Backward

Let us define four scalar inputs and compute a composite mathematical expression:
$$L = (a \cdot b + c)^2 / 2.0$$

Let $a = 2.0, b = -3.0, c = 10.0$:

In [ ]:
a = Tensor(2.0)
b = Tensor(-3.0)
c = Tensor(10.0)

# Forward operations building the graph
d = a * b         # 2.0 * -3.0 = -6.0
e = d + c         # -6.0 + 10.0 = 4.0
f = e ** 2        # 4.0 ** 2 = 16.0
L = f / 2.0       # 16.0 / 2.0 = 8.0

print(f"Forward Result L = {L.data.item()}")

# Trigger reverse-mode automatic differentiation!
L.backward()

print("\nComputed Gradients:")
print(f"  dL/da = {a.grad.item():.4f}  (Expected: e * b = 4 * -3 = -12.0)")
print(f"  dL/db = {b.grad.item():.4f}  (Expected: e * a = 4 * 2  = 8.0)")
print(f"  dL/dc = {c.grad.item():.4f}  (Expected: e * 1 = 4 * 1  = 4.0)")

## 2. Verifying Autograd Against PyTorch Ground Truth

Let us compute the exact same mathematical graph in PyTorch and compare gradients:

In [ ]:
pt_a = torch.tensor(2.0, requires_grad=True, dtype=torch.float64)
pt_b = torch.tensor(-3.0, requires_grad=True, dtype=torch.float64)
pt_c = torch.tensor(10.0, requires_grad=True, dtype=torch.float64)

pt_d = pt_a * pt_b
pt_e = pt_d + pt_c
pt_f = pt_e ** 2
pt_L = pt_f / 2.0
pt_L.backward()

print(f"{'Variable':<10} | {'Our Autograd':<15} | {'PyTorch':<15} | {'Match?':<10}")
print("-" * 55)
for name, our_g, pt_g in [
    ("dL/da", a.grad.item(), pt_a.grad.item()),
    ("dL/db", b.grad.item(), pt_b.grad.item()),
    ("dL/dc", c.grad.item(), pt_c.grad.item()),
]:
    match = np.isclose(our_g, pt_g)
    print(f"{name:<10} | {our_g:<15.6f} | {pt_g:<15.6f} | {str(match):<10}")

## 3. The Power of Variable Reuse: Multivariable Chain Rule

What happens when a variable is reused in multiple branches of a formula? E.g.:
$$y = x^2 + 3x$$

By calculus: $\frac{dy}{dx} = 2x + 3$. If $x = 4$, $\frac{dy}{dx} = 2(4) + 3 = 11$.
Because our Autograd accumulates gradients with `+=`, gradients from all branches naturally sum together:

In [ ]:
x = Tensor(4.0)
y = x ** 2 + 3.0 * x
y.backward()

print(f"For y = x^2 + 3x at x = 4.0:")
print(f"  y = {y.data.item()}")
print(f"  dy/dx = {x.grad.item()} (Analytical: 2*4 + 3 = 11)")

## 4. Multi-Dimensional Tensors & Matrix Multiplication

Our Autograd engine seamlessly handles multi-dimensional arrays, matrix multiplications (`@`), and automatic broadcasting unsumming for neural network layers:

In [ ]:
X = Tensor(np.random.randn(3, 4))   # Mini-batch of 3 samples, 4 features
W = Tensor(np.random.randn(4, 2))   # Weight matrix (4 -> 2)
b = Tensor(np.zeros((1, 2)))        # Bias vector (broadcast across batch)

out = (X @ W + b).relu()
loss = out.sum()
loss.backward()

print(f"X grad shape: {X.grad.shape} (Matches X: {X.shape})")
print(f"W grad shape: {W.grad.shape} (Matches W: {W.shape})")
print(f"b grad shape: {b.grad.shape} (Matches b: {b.shape})")

## 5. Training an MLP with Zero Manual Gradients

Now let us train a complete 2-layer Multi-Layer Perceptron on a non-linear dataset (two interleaving crescents/moons) completely via our Autograd engine:

In [ ]:
# Generate synthetic 2-moons dataset
np.random.seed(42)
n = 200
t = np.linspace(0, np.pi, n // 2)
X1 = np.stack([np.cos(t), np.sin(t)], axis=1) + np.random.randn(n // 2, 2) * 0.1
X2 = np.stack([1 - np.cos(t), 1 - np.sin(t) - 0.5], axis=1) + np.random.randn(n // 2, 2) * 0.1
X_data = np.vstack([X1, X2])
y_data = np.array([0] * (n // 2) + [1] * (n // 2))

X_tensor = Tensor(X_data)

# Create MLP: 2 inputs -> 16 hidden -> 16 hidden -> 2 classes
model = MLP(in_features=2, hidden_dims=[16, 16], out_features=2, seed=42)
lr = 0.2
losses = []

for ep in range(1, 101):
    logits = model(X_tensor)
    loss = cross_entropy_loss(logits, y_data)
    
    model.zero_grad()
    loss.backward()
    
    for p in model.parameters():
        p.data -= lr * p.grad
        
    losses.append(loss.data.item())
    if ep % 20 == 0:
        preds = np.argmax(logits.data, axis=1)
        acc = np.mean(preds == y_data) * 100
        print(f"Epoch {ep:3d} | Loss: {loss.data.item():.4f} | Accuracy: {acc:.1f}%")

## 6. Visualizing the Non-Linear Decision Boundary

In [ ]:
xx, yy = np.meshgrid(np.linspace(X_data[:, 0].min() - 0.5, X_data[:, 0].max() + 0.5, 150),
                     np.linspace(X_data[:, 1].min() - 0.5, X_data[:, 1].max() + 0.5, 150))
grid = np.c_[xx.ravel(), yy.ravel()]
grid_preds = np.argmax(model(Tensor(grid)).data, axis=1).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, grid_preds, cmap="coolwarm", alpha=0.3)
plt.scatter(X_data[y_data == 0, 0], X_data[y_data == 0, 1], c="#2563eb", edgecolors="k", label="Class 0")
plt.scatter(X_data[y_data == 1, 0], X_data[y_data == 1, 1], c="#dc2626", edgecolors="k", label="Class 1")
plt.title("Non-Linear Decision Boundary Learned via Pure Autograd", fontweight="bold")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()